[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CalculatedContent/xgboost2ww/blob/main/notebooks/XGBWW_Random10_LongRun_Alpha_Tracking.ipynb)

# Random-10 Dataset Long-Run Alpha Tracking

**Designed for Google Colab.** This notebook is Colab-first: it mounts Google Drive for persistent checkpointing, installs `xgboost2ww` and `xgbwwdata` from source, resumes long runs from Drive checkpoints, and prefers Colab GPU when available (with CPU `hist` fallback).

## 1. Colab / Drive setup

In [1]:
# Colab-first runtime + Drive setup
import os
import sys
import platform
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False

USE_GOOGLE_DRIVE = True if IN_COLAB else False
FORCE_REMOUNT_DRIVE = False

print(f"IN_COLAB={IN_COLAB}")
print(f"Python={sys.version.split()[0]}")
print(f"Platform={platform.platform()}")

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=FORCE_REMOUNT_DRIVE)
    project_root = Path('/content/drive/MyDrive/xgboost2ww_runs/random10_longrun_alpha_tracking')
else:
    project_root = Path('./random10_longrun_alpha_tracking')

registry_dir = project_root / 'registry'
per_dataset_dir = project_root / 'per_dataset'
aggregate_dir = project_root / 'aggregate'
logs_dir = project_root / 'logs'
for p in [project_root, registry_dir, per_dataset_dir, aggregate_dir, logs_dir]:
    p.mkdir(parents=True, exist_ok=True)

errors_path = logs_dir / 'errors.csv'
skipped_path = aggregate_dir / 'skipped_datasets.csv'

if IN_COLAB:
    print('Drive mounted')
    print('Google Drive checkpointing enabled')

print('project_root:', project_root)
print('registry_dir:', registry_dir)
print('per_dataset_dir:', per_dataset_dir)
print('aggregate_dir:', aggregate_dir)
print('logs_dir:', logs_dir)


IN_COLAB=True
Python=3.12.12
Platform=Linux-6.6.113+-x86_64-with-glibc2.35
Mounted at /content/drive
Drive mounted
Google Drive checkpointing enabled
project_root: /content/drive/MyDrive/xgboost2ww_runs/random10_longrun_alpha_tracking
registry_dir: /content/drive/MyDrive/xgboost2ww_runs/random10_longrun_alpha_tracking/registry
per_dataset_dir: /content/drive/MyDrive/xgboost2ww_runs/random10_longrun_alpha_tracking/per_dataset
aggregate_dir: /content/drive/MyDrive/xgboost2ww_runs/random10_longrun_alpha_tracking/aggregate
logs_dir: /content/drive/MyDrive/xgboost2ww_runs/random10_longrun_alpha_tracking/logs


## 2. Colab/bootstrap installs

In [2]:
# Colab/bootstrap installs
%pip install -q xgboost weightwatcher scikit-learn pandas matplotlib seaborn scipy feather-format pyarrow
%pip install -q openml pmlb keel-ds
!git clone --depth 1 https://github.com/CalculatedContent/xgboost2ww.git /tmp/xgboost2ww-src || true
!git clone --depth 1 https://github.com/CalculatedContent/xgbwwdata.git /tmp/xgbwwdata-src || true
%pip install -q /tmp/xgboost2ww-src
%pip install -q /tmp/xgbwwdata-src

import importlib
import pathlib

xgboost = importlib.import_module('xgboost')
weightwatcher = importlib.import_module('weightwatcher')
xgboost2ww = importlib.import_module('xgboost2ww')
xgbwwdata = importlib.import_module('xgbwwdata')

print('xgboost:', getattr(xgboost, '__version__', 'unknown'))
print('weightwatcher:', getattr(weightwatcher, '__version__', 'unknown'))
print('xgboost2ww location:', pathlib.Path(xgboost2ww.__file__).resolve())
print('xgbwwdata location:', pathlib.Path(xgbwwdata.__file__).resolve())
print('xgboost2ww installed from source')
print('xgbwwdata installed from source')


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.7/83.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.0/192.0 kB 8.1 MB/s eta 0:00:00
Cloning into '/tmp/xgboost2ww-src'...
remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 48 (delta 4), reused 35 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (48/48), 7.05 MiB | 19.29 MiB/s, done.
Resolving deltas: 100% (4/4), done.
Cloning into '/tmp/xgbwwdata-src'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 32 (delta 0), reused 18 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (32/32), 174.03 KiB | 5.61 MiB/s, done.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing bui

xgboost: 3.2.0
weightwatcher: 0.7.7
xgboost2ww location: /usr/local/lib/python3.12/dist-packages/xgboost2ww/__init__.py
xgbwwdata location: /usr/local/lib/python3.12/dist-packages/xgbwwdata/__init__.py
xgboost2ww installed from source
xgbwwdata installed from source


If installs are rerun many times and import state becomes inconsistent, restart runtime and rerun from the top. In normal usage, this notebook is structured to avoid requiring a manual restart.

## 3. Imports

In [3]:
# Imports
import json
import time
import random
import traceback
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import weightwatcher as ww

from scipy import sparse
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

from xgboost2ww import convert
from xgbwwdata import Filters, load_dataset

try:
    from xgbwwdata import scan_datasets as xgbww_scan_datasets
except Exception:
    xgbww_scan_datasets = None

try:
    from xgbwwdata import list_datasets as xgbww_list_datasets
except Exception:
    xgbww_list_datasets = None


## 4. Runtime configuration

Long Colab runs may disconnect. This notebook is checkpointed to Drive so you can reconnect and rerun from the top to resume. Use `FORCE_RESTART_ALL=True` to start over, or set `FORCE_RESTART_DATASETS` to selected dataset slugs/uids.

In [4]:
# Runtime + global config
RANDOM_STATE = 42
DATASET_SAMPLE_SIZE = 10
TOTAL_ROUNDS = 1200
CHUNK_SIZE = 25
N_STEPS = TOTAL_ROUNDS // CHUNK_SIZE
CHECKPOINT_EVERY_STEPS = 1

RESUME_FROM_CHECKPOINT = True
FORCE_RESTART_ALL = False
FORCE_RESTART_DATASETS = []
REUSE_SAMPLED_REGISTRY = True

SELECTED_DATASET_UIDS = []
DRY_RUN = False
MAX_DENSE_ELEMENTS = int(2e8)

if DRY_RUN:
    DATASET_SAMPLE_SIZE = 2
    TOTAL_ROUNDS = 50
    CHUNK_SIZE = 25
    N_STEPS = TOTAL_ROUNDS // CHUNK_SIZE

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
sns.set_theme(style='whitegrid')
print(f'N_STEPS={N_STEPS}, sample={DATASET_SAMPLE_SIZE}, dry_run={DRY_RUN}')
print(f'USE_GOOGLE_DRIVE={USE_GOOGLE_DRIVE}, RESUME_FROM_CHECKPOINT={RESUME_FROM_CHECKPOINT}, REUSE_SAMPLED_REGISTRY={REUSE_SAMPLED_REGISTRY}')


N_STEPS=48, sample=10, dry_run=False
USE_GOOGLE_DRIVE=True, RESUME_FROM_CHECKPOINT=True, REUSE_SAMPLED_REGISTRY=True


## 5. Runtime diagnostics

In [5]:
# Runtime diagnostics + Colab-aware backend selection
import warnings

print('cpu_count:', os.cpu_count())
print('platform:', platform.platform())

if IN_COLAB:
    try:
        smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True, check=False)
        print('nvidia-smi returncode:', smi.returncode)
        print(smi.stdout[:1000] if smi.stdout else smi.stderr[:500])
    except Exception as e:
        print('[WARN] nvidia-smi unavailable:', e)

def _probe_xgb(params_patch):
    X_probe = np.array([[0.0], [1.0], [2.0], [3.0]], dtype=np.float32)
    y_probe = np.array([0, 0, 1, 1], dtype=np.float32)
    dprobe = xgb.DMatrix(X_probe, label=y_probe)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        xgb.train(params={'objective': 'binary:logistic', 'eval_metric': 'logloss', **params_patch}, dtrain=dprobe, num_boost_round=1, verbose_eval=False)
    gpu_fallback_msgs = [
        str(w.message) for w in caught
        if ('No visible GPU is found' in str(w.message)) or ('Device is changed from GPU to CPU' in str(w.message))
    ]
    if gpu_fallback_msgs:
        raise RuntimeError(' ; '.join(gpu_fallback_msgs))

def choose_xgb_runtime_colab_aware():
    cpu_threads = max(1, (os.cpu_count() or 1) - 1)
    print(f'colab_gpu_runtime={IN_COLAB}')

    if IN_COLAB:
        try:
            modern_gpu = {'tree_method': 'hist', 'device': 'cuda'}
            _probe_xgb(modern_gpu)
            print('CUDA visible to XGBoost: yes')
            print('Backend fallback path: modern gpu config succeeded')
            return {'backend_name': 'colab_cuda_hist', 'params_patch': modern_gpu, 'is_gpu': True}
        except Exception as e1:
            print('CUDA visible to XGBoost: no (modern config failed)')
            print('[fallback] modern GPU failed -> trying gpu_hist:', e1)
            try:
                legacy_gpu = {'tree_method': 'gpu_hist'}
                _probe_xgb(legacy_gpu)
                print('Backend fallback path: legacy gpu_hist succeeded')
                return {'backend_name': 'colab_gpu_hist_legacy', 'params_patch': legacy_gpu, 'is_gpu': True}
            except Exception as e2:
                print('[fallback] gpu_hist failed -> using CPU hist:', e2)

    cpu_patch = {'tree_method': 'hist', 'nthread': cpu_threads}
    print('Backend fallback path: CPU hist')
    return {'backend_name': 'cpu_hist', 'params_patch': cpu_patch, 'is_gpu': False}

BACKEND_INFO = choose_xgb_runtime_colab_aware()
print('Chosen backend:', BACKEND_INFO['backend_name'])
print('Chosen XGBoost params patch:', BACKEND_INFO['params_patch'])



cpu_count: 2
platform: Linux-6.6.113+-x86_64-with-glibc2.35
[WARN] nvidia-smi unavailable: [Errno 2] No such file or directory: 'nvidia-smi'
colab_gpu_runtime=True
CUDA visible to XGBoost: yes
Backend fallback path: modern gpu config succeeded
Chosen backend: colab_cuda_hist
Chosen XGBoost params patch: {'tree_method': 'hist', 'device': 'cuda'}


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [00:10:26] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [00:10:26] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)


## 6. Dataset catalog scan / cached registry

In [6]:
# xgbwwdata scan / registry build

import inspect
import importlib
import sys
from dataclasses import asdict, is_dataclass


def _install_missing_dataset_dependency(module_name: str) -> bool:
    if module_name != 'keel_ds':
        return False

    print('[INFO] Missing optional dataset dependency: keel_ds. Installing now...')
    candidates = ['keel-ds', 'keel_ds']
    for pkg in candidates:
        try:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
            importlib.import_module('keel_ds')
            print(f'[INFO] Installed KEEL dependency using package: {pkg}')
            return True
        except Exception:
            continue

    print('[WARN] Unable to auto-install keel_ds. KEEL datasets will be skipped for this run.')
    return False


def _copy_filters_with_overrides(filters: Filters, **overrides):
    if is_dataclass(filters):
        data = asdict(filters)
    elif hasattr(filters, 'model_dump'):
        data = filters.model_dump()
    elif hasattr(filters, '__dict__'):
        data = dict(filters.__dict__)
    else:
        data = {}
    data.update(overrides)
    return type(filters)(**data)


def _filters_without_keel(filters: Filters):
    source_keys = ('sources', 'include_sources', 'source_allowlist')
    for key in source_keys:
        current = getattr(filters, key, None)
        if not current:
            continue
        try:
            cleaned = [s for s in current if 'keel' not in str(s).lower()]
            if cleaned and len(cleaned) != len(current):
                print(f'[INFO] Retrying scan without KEEL via filters.{key}.')
                return _copy_filters_with_overrides(filters, **{key: cleaned})
        except Exception:
            continue
    return None


def _call_xgbww_scan(api_func, filters: Filters):
    sig = inspect.signature(api_func)
    kwargs = {}
    if 'filters' in sig.parameters:
        kwargs['filters'] = filters
    if 'preprocess' in sig.parameters:
        kwargs['preprocess'] = True
    if 'smoke_train' in sig.parameters:
        kwargs['smoke_train'] = True
    if not kwargs:
        return api_func()
    return api_func(**kwargs)


def _safe_call_scan(api_func, filters: Filters):
    try:
        return _call_xgbww_scan(api_func, filters)
    except ModuleNotFoundError as e:
        missing = getattr(e, 'name', None)
        if not missing:
            raise

        installed = _install_missing_dataset_dependency(missing)
        if installed:
            return _call_xgbww_scan(api_func, filters)

        if missing == 'keel_ds':
            fallback_filters = _filters_without_keel(filters)
            if fallback_filters is not None:
                return _call_xgbww_scan(api_func, fallback_filters)

        raise


def build_registry_df(filters: Filters) -> pd.DataFrame:
    if xgbww_scan_datasets is not None:
        reg = _safe_call_scan(xgbww_scan_datasets, filters)
        return pd.DataFrame(reg)
    if xgbww_list_datasets is not None:
        reg = _safe_call_scan(xgbww_list_datasets, filters)
        return pd.DataFrame(reg)
    try:
        from xgbwwdata import get_registry
        reg = _safe_call_scan(get_registry, filters)
        return pd.DataFrame(reg)
    except Exception as e:
        raise RuntimeError('No compatible xgbwwdata scan/list API found in this environment.') from e


filters = Filters(min_rows=200, max_rows=60000, max_features=50000, max_dense_elements=MAX_DENSE_ELEMENTS, preprocess=True)
full_registry_csv = registry_dir / 'full_registry.csv'
full_registry_feather = registry_dir / 'full_registry.feather'

if RESUME_FROM_CHECKPOINT and full_registry_csv.exists() and not FORCE_RESTART_ALL:
    full_registry_df = pd.read_csv(full_registry_csv)
    print(f'Loaded cached full registry: {len(full_registry_df)} rows from {full_registry_csv}')
else:
    full_registry_df = build_registry_df(filters)
    full_registry_df.to_csv(full_registry_csv, index=False)
    try:
        full_registry_df.reset_index(drop=True).to_feather(full_registry_feather)
    except Exception as e:
        print('[WARN] Feather save failed for full registry:', e)
    print(f'Scanned full registry: {len(full_registry_df)} rows')

registry_df = full_registry_df.copy()
if 'task' in registry_df.columns:
    registry_df = registry_df[registry_df['task'].astype(str).str.contains('class', case=False, na=False)]
if 'n_classes' in registry_df.columns:
    registry_df = registry_df[(registry_df['n_classes'].fillna(0) >= 2) & (registry_df['n_classes'].fillna(0) <= 20)]
registry_df = registry_df.reset_index(drop=True)
print('Filtered registry rows:', len(registry_df))


TypeError: scan_datasets() got an unexpected keyword argument 'preprocess'

## 7. Random-10 dataset selection

In [ ]:
# Random sample of 10 datasets (saved immediately to Drive in Colab)
random10_csv = registry_dir / 'random10_registry.csv'
random10_feather = registry_dir / 'random10_registry.feather'

if SELECTED_DATASET_UIDS:
    sampled_registry = registry_df[registry_df['dataset_uid'].isin(SELECTED_DATASET_UIDS)].copy()
    print(f'Using explicit SELECTED_DATASET_UIDS ({len(sampled_registry)} rows).')
elif REUSE_SAMPLED_REGISTRY and RESUME_FROM_CHECKPOINT and random10_csv.exists() and not FORCE_RESTART_ALL:
    sampled_registry = pd.read_csv(random10_csv)
    print(f'Reusing sampled registry from checkpoint: {len(sampled_registry)} rows ({random10_csv})')
else:
    n_take = min(DATASET_SAMPLE_SIZE, len(registry_df))
    sampled_registry = registry_df.sample(n=n_take, random_state=RANDOM_STATE).copy()
    print(f'Sampled {len(sampled_registry)} datasets')

sampled_registry = sampled_registry.reset_index(drop=True)
sampled_registry.to_csv(random10_csv, index=False)
try:
    sampled_registry.reset_index(drop=True).to_feather(random10_feather)
except Exception as e:
    print('[WARN] Feather save failed for sampled registry:', e)
print('Saved sampled registry to:', random10_csv)

display(sampled_registry.head(20))


## 8. Helper functions

In [ ]:
# Helper functions
W_LIST = ['W1', 'W2', 'W7', 'W8', 'W9', 'W10']

def make_safe_slug(text):
    text = str(text)
    out = ''.join(ch.lower() if ch.isalnum() else '_' for ch in text)
    while '__' in out:
        out = out.replace('__', '_')
    return out.strip('_')[:120]

def log_error(error_rows, dataset_uid, dataset_slug, stage, exc):
    error_rows.append({'dataset_uid': dataset_uid, 'dataset_slug': dataset_slug, 'stage': stage, 'error_type': type(exc).__name__, 'error_message': str(exc), 'traceback': traceback.format_exc(), 'timestamp': pd.Timestamp.utcnow().isoformat()})

def save_error_log(error_rows, errors_path):
    if error_rows:
        pd.DataFrame(error_rows).to_csv(errors_path, index=False)

def make_dataset_paths(dataset_slug):
    base = per_dataset_dir / dataset_slug
    out = {'base': base, 'results': base / 'results', 'models': base / 'models', 'plots': base / 'plots'}
    for p in out.values():
        p.mkdir(parents=True, exist_ok=True)
    return out

def detect_task_type(y):
    y_np = np.asarray(y)
    n_classes = int(len(np.unique(y_np)))
    if n_classes < 2:
        return 'invalid', n_classes, y_np
    return ('binary' if n_classes == 2 else 'multiclass'), n_classes, y_np

def build_params_for_dataset(base_backend_patch, n_classes):
    params = {'max_depth': 4, 'eta': 0.05, 'subsample': 0.9, 'colsample_bytree': 0.8, 'min_child_weight': 5, 'reg_lambda': 5.0, 'reg_alpha': 0.5, 'gamma': 1.0, 'seed': RANDOM_STATE, **base_backend_patch}
    if n_classes == 2:
        params.update({'objective': 'binary:logistic', 'eval_metric': 'logloss'})
    else:
        params.update({'objective': 'multi:softprob', 'eval_metric': 'mlogloss', 'num_class': int(n_classes)})
    return params

def prepare_train_test_data(X, y):
    stratify = y if len(np.unique(y)) > 1 else None
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=stratify)
    scaler = StandardScaler(with_mean=False) if sparse.issparse(X_train) else StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
    X_test_scaled = scaler.transform(X_test).astype(np.float32)
    y_train_np = np.asarray(y_train).astype(np.int32).reshape(-1)
    y_test_np = np.asarray(y_test).astype(np.int32).reshape(-1)
    return X_train_scaled, X_test_scaled, y_train_np, y_test_np

def densify_for_convert_if_safe(X_train_scaled, max_dense_elements):
    if sparse.issparse(X_train_scaled):
        n_elements = int(X_train_scaled.shape[0]) * int(X_train_scaled.shape[1])
        if n_elements > max_dense_elements:
            raise MemoryError(f'Refusing to densify convert input: shape={X_train_scaled.shape}, elements={n_elements:,}')
        return X_train_scaled.toarray().astype(np.float32)
    return np.asarray(X_train_scaled).astype(np.float32)

def build_layer_for_W(bst, W_name, current_round, X_train_for_convert, y_train_np, params):
    multiclass_mode = 'softprob' if int(params.get('num_class', 2)) > 2 else 'error'
    return convert(model=bst, data=X_train_for_convert, labels=y_train_np, W=W_name, return_type='torch', nfolds=5, t_points=min(current_round, 160), random_state=RANDOM_STATE, train_params=params, num_boost_round=current_round, multiclass=multiclass_mode)

def compute_alpha_from_layer(layer):
    watcher = ww.WeightWatcher(model=layer)
    df = watcher.analyze(randomize=True, detX=True)
    return float(df['alpha'].iloc[0])

def save_dataset_checkpoint(rows, metrics_csv_path, metrics_feather_path, bst, current_round, models_dir, metadata, meta_json_path):
    df = pd.DataFrame(rows)
    df.to_csv(metrics_csv_path, index=False)
    try:
        df.reset_index(drop=True).to_feather(metrics_feather_path)
    except Exception as e:
        print('[WARN] Feather save failed:', e)

    round_model_path = models_dir / f'model_round_{current_round:04d}.json'
    latest_model_path = models_dir / 'model_latest.json'
    bst.save_model(round_model_path)
    bst.save_model(latest_model_path)

    metadata = dict(metadata)
    metadata['latest_completed_round'] = int(current_round)
    metadata['updated_at'] = pd.Timestamp.utcnow().isoformat()
    meta_json_path.write_text(json.dumps(metadata, indent=2))

    print(f"[CHECKPOINT] round={current_round} saved -> {metrics_csv_path}")
    print(f"[CHECKPOINT] model_latest -> {latest_model_path}")

def load_dataset_checkpoint(metrics_csv_path, latest_model_path, chunk_size, n_steps):
    if not (metrics_csv_path.exists() and latest_model_path.exists()):
        return None, 1, []
    prior_df = pd.read_csv(metrics_csv_path)
    if prior_df.empty:
        return None, 1, []
    max_round = int(prior_df['boosting_round'].max())
    if max_round % chunk_size != 0:
        raise ValueError(f'Checkpoint round {max_round} incompatible with CHUNK_SIZE={chunk_size}')
    completed_steps = max_round // chunk_size
    if completed_steps > n_steps:
        raise ValueError(f'Checkpoint completed_steps={completed_steps} > N_STEPS={n_steps}')
    bst = xgb.Booster()
    bst.load_model(latest_model_path)
    return bst, completed_steps + 1, prior_df.to_dict('records')

def plot_dataset_dynamics(df_dataset, dataset_slug, plots_dir):
    if df_dataset.empty:
        return None, None
    long_alpha = df_dataset.melt(id_vars=['boosting_round', 'test_accuracy'], value_vars=[f'alpha_{w}' for w in W_LIST], var_name='alpha_name', value_name='alpha_value')
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    axes[0].plot(df_dataset['boosting_round'], df_dataset['test_accuracy'], marker='o')
    axes[0].set_title(f'{dataset_slug}: test accuracy vs round')
    sns.lineplot(data=long_alpha, x='boosting_round', y='alpha_value', hue='alpha_name', ax=axes[1])
    axes[1].set_title('alpha vs round')
    sns.scatterplot(data=long_alpha, x='alpha_value', y='test_accuracy', hue='alpha_name', ax=axes[2])
    axes[2].set_title('accuracy vs alpha')
    fig.tight_layout()
    p1 = plots_dir / f'{dataset_slug}_dynamics.png'
    fig.savefig(p1, dpi=140)
    plt.close(fig)
    fig2, ax2 = plt.subplots(figsize=(7, 5))
    for w in W_LIST:
        col = f'alpha_{w}'
        ax2.scatter(df_dataset[col], df_dataset['test_accuracy'], alpha=0.6, label=w)
    ax2.set_title(f'{dataset_slug}: accuracy vs alpha (all W)')
    ax2.legend(ncol=2)
    fig2.tight_layout()
    p2 = plots_dir / f'{dataset_slug}_accuracy_vs_alpha.png'
    fig2.savefig(p2, dpi=140)
    plt.close(fig2)
    return p1, p2


## 9. Per-dataset long-run training

In [ ]:
# Per-dataset training loop
all_results, skipped_rows, error_rows = [], [], []

for i, row in sampled_registry.iterrows():
    dataset_uid = row.get('dataset_uid', f'row_{i}')
    source = row.get('source', 'unknown')
    dataset_slug = make_safe_slug(f"{dataset_uid}_{source}")
    print(f"\n[{i+1}/{len(sampled_registry)}] dataset_uid={dataset_uid} source={source}")
    force_restart_this = FORCE_RESTART_ALL or (dataset_slug in FORCE_RESTART_DATASETS) or (dataset_uid in FORCE_RESTART_DATASETS)

    paths = make_dataset_paths(dataset_slug)
    metrics_csv = paths['results'] / f'{dataset_slug}_metrics.csv'
    metrics_feather = paths['results'] / f'{dataset_slug}_metrics.feather'
    latest_model_path = paths['models'] / 'model_latest.json'
    dataset_meta_json = paths['results'] / '_meta.json'

    try:
        X, y, meta = load_dataset(dataset_uid, filters=filters)
    except Exception as e:
        log_error(error_rows, dataset_uid, dataset_slug, 'dataset_load', e)
        skipped_rows.append({'dataset_uid': dataset_uid, 'dataset_slug': dataset_slug, 'reason': 'dataset_load_failed'})
        save_error_log(error_rows, errors_path)
        continue

    try:
        task_type, n_classes, y_np = detect_task_type(y)
        if task_type == 'invalid' or n_classes > 20:
            raise ValueError(f'Invalid class count: n_classes={n_classes}')
        X_train_scaled, X_test_scaled, y_train_np, y_test_np = prepare_train_test_data(X, y_np)
        X_train_for_convert = densify_for_convert_if_safe(X_train_scaled, MAX_DENSE_ELEMENTS)
    except Exception as e:
        log_error(error_rows, dataset_uid, dataset_slug, 'split_preprocess', e)
        skipped_rows.append({'dataset_uid': dataset_uid, 'dataset_slug': dataset_slug, 'reason': 'split_preprocess_failed'})
        save_error_log(error_rows, errors_path)
        continue

    dtrain, dtest = xgb.DMatrix(X_train_scaled, label=y_train_np), xgb.DMatrix(X_test_scaled, label=y_test_np)
    params = build_params_for_dataset(BACKEND_INFO['params_patch'], n_classes)

    base_meta = {
        'dataset_uid': str(dataset_uid),
        'dataset_slug': dataset_slug,
        'source': str(source),
        'shapes': {
            'X_train': [int(X_train_scaled.shape[0]), int(X_train_scaled.shape[1])],
            'X_test': [int(X_test_scaled.shape[0]), int(X_test_scaled.shape[1])],
        },
        'class_count': int(n_classes),
        'selected_params': params,
        'backend_info': BACKEND_INFO,
        'start_time_utc': pd.Timestamp.utcnow().isoformat(),
        'latest_completed_round': 0,
    }
    dataset_meta_json.write_text(json.dumps(base_meta, indent=2))

    bst, rows, start_step = None, [], 1
    if (not force_restart_this) and RESUME_FROM_CHECKPOINT:
        try:
            bst, start_step, rows = load_dataset_checkpoint(metrics_csv, latest_model_path, CHUNK_SIZE, N_STEPS)
            if bst is not None:
                print(f'[RESUME] step={start_step}/{N_STEPS} from {metrics_csv}')
        except Exception as e:
            log_error(error_rows, dataset_uid, dataset_slug, 'checkpoint_load', e)
            bst, rows, start_step = None, [], 1

    dataset_t0 = time.time()
    dataset_failed = False
    for step in range(start_step, N_STEPS + 1):
        current_round = step * CHUNK_SIZE
        step_t0 = time.time()
        try:
            bst = xgb.train(params=params, dtrain=dtrain, num_boost_round=CHUNK_SIZE, xgb_model=bst, verbose_eval=False)
            y_prob = bst.predict(dtest)
            y_pred = (y_prob >= 0.5).astype(np.int32) if n_classes == 2 else np.argmax(y_prob, axis=1).astype(np.int32)
            test_accuracy = float(accuracy_score(y_test_np, y_pred))
        except Exception as e:
            log_error(error_rows, dataset_uid, dataset_slug, 'train_or_predict', e)
            dataset_failed = True
            break

        alpha_values, alpha_failures = {}, 0
        for w_name in W_LIST:
            try:
                layer = build_layer_for_W(bst, w_name, current_round, X_train_for_convert, y_train_np, params)
                alpha_values[f'alpha_{w_name}'] = compute_alpha_from_layer(layer)
            except Exception as e:
                alpha_values[f'alpha_{w_name}'] = np.nan
                alpha_failures += 1
                log_error(error_rows, dataset_uid, dataset_slug, f'alpha_{w_name}', e)

        if alpha_failures == len(W_LIST):
            dataset_failed = True
            log_error(error_rows, dataset_uid, dataset_slug, 'alpha_all_failed', RuntimeError('All W matrices failed this round'))
            break

        rows.append({'dataset_uid': dataset_uid, 'dataset_slug': dataset_slug, 'source': source, 'n_samples': int(np.asarray(y_np).shape[0]), 'n_features': int(X_train_scaled.shape[1]), 'n_classes': int(n_classes), 'boosting_round': int(current_round), 'test_accuracy': test_accuracy, **alpha_values, 'backend': BACKEND_INFO.get('backend_name'), 'tree_method': params.get('tree_method'), 'device': params.get('device', 'cpu'), 'elapsed_seconds_round': float(time.time() - step_t0), 'elapsed_seconds_total': float(time.time() - dataset_t0)})

        if step % CHECKPOINT_EVERY_STEPS == 0:
            try:
                save_dataset_checkpoint(rows, metrics_csv, metrics_feather, bst, current_round, paths['models'], base_meta, dataset_meta_json)
            except Exception as e:
                log_error(error_rows, dataset_uid, dataset_slug, 'checkpoint_save', e)

        print(f"  step={step:03d}/{N_STEPS} round={current_round:04d} acc={test_accuracy:.4f}")

    df_dataset = pd.DataFrame(rows)
    if not df_dataset.empty:
        all_results.append(df_dataset)
        try:
            plot_dataset_dynamics(df_dataset, dataset_slug, paths['plots'])
        except Exception as e:
            log_error(error_rows, dataset_uid, dataset_slug, 'plotting', e)
    if dataset_failed and df_dataset.empty:
        skipped_rows.append({'dataset_uid': dataset_uid, 'dataset_slug': dataset_slug, 'reason': 'dataset_failed'})

    save_error_log(error_rows, errors_path)

print('Done processing sampled datasets.')


## 10. Per-dataset plots

Per-dataset alpha/accuracy plots are saved under each dataset folder in `per_dataset/<dataset_slug>/plots/` on Drive when running in Colab.

## 11. Aggregate summaries

In [ ]:
# Aggregate summary / plots
all_df = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()

all_csv = aggregate_dir / 'all_datasets_longrun_metrics.csv'
all_feather = aggregate_dir / 'all_datasets_longrun_metrics.feather'
best_csv = aggregate_dir / 'best_rows_per_dataset.csv'
corr_csv = aggregate_dir / 'correlation_summary.csv'
run_summary_json = aggregate_dir / 'run_summary.json'

if not all_df.empty:
    all_df.to_csv(all_csv, index=False)
    try:
        all_df.reset_index(drop=True).to_feather(all_feather)
    except Exception as e:
        print('[WARN] Feather save failed for aggregate metrics:', e)

    best_df = all_df.loc[all_df.groupby('dataset_uid')['test_accuracy'].idxmax()].sort_values('test_accuracy', ascending=False)
    best_df.to_csv(best_csv, index=False)

    corr_rows = []
    for dataset_uid, g in all_df.groupby('dataset_uid'):
        row = {'dataset_uid': dataset_uid, 'dataset_slug': g['dataset_slug'].iloc[0]}
        for w in W_LIST:
            c = g['test_accuracy'].corr(g[f'alpha_{w}'])
            row[f'corr_test_accuracy_alpha_{w}'] = float(c) if pd.notnull(c) else np.nan
        corr_rows.append(row)
    corr_df = pd.DataFrame(corr_rows)
    corr_df.to_csv(corr_csv, index=False)

    rank_rows = []
    for w in W_LIST:
        cvals = corr_df[f'corr_test_accuracy_alpha_{w}'].dropna()
        rank_rows.append({'W': w, 'avg_abs_corr': float(np.abs(cvals).mean()) if len(cvals) else np.nan, 'share_lower_alpha_better': float((cvals < 0).mean()) if len(cvals) else np.nan})
    rank_df = pd.DataFrame(rank_rows).sort_values('avg_abs_corr', ascending=False)

    run_summary = {'project_root': str(project_root), 'n_sampled_datasets': int(len(sampled_registry)), 'n_completed_datasets': int(all_df['dataset_uid'].nunique()), 'n_rows_total': int(len(all_df)), 'backend': BACKEND_INFO, 'timestamp_utc': pd.Timestamp.utcnow().isoformat()}
    run_summary_json.write_text(json.dumps(run_summary, indent=2))

    if skipped_rows:
        pd.DataFrame(skipped_rows).to_csv(skipped_path, index=False)

    display(sampled_registry)
    display(pd.DataFrame(skipped_rows) if skipped_rows else pd.DataFrame(columns=['dataset_uid', 'dataset_slug', 'reason']))
    display(best_df)
    display(all_df.sort_values('test_accuracy', ascending=False).head(20))
    display(corr_df)
    display(rank_df)
else:
    print('No completed dataset results to aggregate.')


## 12. Saved artifacts on Google Drive

In [ ]:
# Drive artifact summary
print('Sampled registry CSV:', registry_dir / 'random10_registry.csv')
print('Sampled registry Feather:', registry_dir / 'random10_registry.feather')
print('Error log:', errors_path)
print('Aggregate metrics:', aggregate_dir / 'all_datasets_longrun_metrics.csv')
print('Best rows summary:', aggregate_dir / 'best_rows_per_dataset.csv')
print('Per-dataset folders:', per_dataset_dir)


In [ ]:
# Optional zip export for Colab users (aggregate + logs)
import shutil
import tempfile

zip_output = aggregate_dir / 'random10_longrun_alpha_tracking_outputs.zip'
with tempfile.TemporaryDirectory() as td:
    td_path = Path(td)
    shutil.copytree(aggregate_dir, td_path / 'aggregate', dirs_exist_ok=True)
    shutil.copytree(logs_dir, td_path / 'logs', dirs_exist_ok=True)
    built = shutil.make_archive(str(zip_output).replace('.zip', ''), 'zip', root_dir=td_path)
print('Saved zip:', built)
